# Literature Scanning - Step 2: Fetching papers from OpenAlex

In [1]:
from discovery_utils.getters import openalex
from discovery_utils import PROJECT_DIR, logging
import pandas as pd
from itertools import combinations, chain
from itertools import product

In [7]:
# create search terms for variations on "parenting programme"
parent_support_terms = [
    "parenting", "parent", "caregiver", "family", "families",
]

program_keywords = ["engagement", "engage", "intervention", "program", "programme"]

parenting_program_keywords = [f"({p} AND {pr})" for p, pr in product(parent_support_terms, program_keywords)]

combined_parent_keywords = " OR ".join(parenting_program_keywords)

# create search terms for the specified areas of interest
domain_keywords = [
    # Parenting Programmes  which promote children’s early speech, language and communication 
    "speech OR language OR communication",
    # Parenting Programmes which promote children’s early cognitive development
    "cognitive development",
    # Parenting Programmes which promote children’s early personal, social, emotional and behavioral development
    "personal OR social OR emotional",
    # Parenting Programmes which promote children’s physical health and development
    "physical OR health",
    # Programmes which support parents’ confidence, positive behaviours, and skills levels as a parent (including parental self-efficacy)
    "self-efficacy OR confidence OR parenting skills",
    # Programmes which support positive changes in parents beliefs around parenting and parental knowledge  
    "parenting attitudes OR parenting beliefs",
    # Programmes which support parents wellbeing and health
    "parental wellbeing OR maternal wellbeing OR parental health OR maternal health OR parental mental health OR maternal mental health",]

queries = []

for kw in domain_keywords:
    queries.append(f"({combined_parent_keywords}) AND ({kw})")

In [8]:
queries

['((parenting AND engagement) OR (parenting AND engage) OR (parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND engagement) OR (parent AND engage) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND engagement) OR (caregiver AND engage) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme) OR (family AND engagement) OR (family AND engage) OR (family AND intervention) OR (family AND program) OR (family AND programme) OR (families AND engagement) OR (families AND engage) OR (families AND intervention) OR (families AND program) OR (families AND programme)) AND (speech OR language OR communication)',
 '((parenting AND engagement) OR (parenting AND engage) OR (parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND engagement) OR (parent AND engage) OR (parent AND intervention) OR (parent AND program) OR (parent AND programm

In [9]:
data_dfs = []

index = 0

for query in queries:
    logging.info(f"Querying OpenAlex for '{query}'")
    data_df = (
        openalex.get_openalex_works(query, n_works = 10000)
        .assign(query=query)
    )
    data_df.to_csv(f'query_{index}.csv', index=False)
    index += 1
    logging.info(f"Collected {len(data_df)} works")
    data_dfs.append(data_df)
    
data_df = pd.concat(data_dfs, ignore_index=True)

2025-05-16 15:46:21,034 - root - INFO - Querying OpenAlex for '((parenting AND engagement) OR (parenting AND engage) OR (parenting AND intervention) OR (parenting AND program) OR (parenting AND programme) OR (parent AND engagement) OR (parent AND engage) OR (parent AND intervention) OR (parent AND program) OR (parent AND programme) OR (caregiver AND engagement) OR (caregiver AND engage) OR (caregiver AND intervention) OR (caregiver AND program) OR (caregiver AND programme) OR (family AND engagement) OR (family AND engage) OR (family AND intervention) OR (family AND program) OR (family AND programme) OR (families AND engagement) OR (families AND engage) OR (families AND intervention) OR (families AND program) OR (families AND programme)) AND (speech OR language OR communication)'
2025-05-16 15:49:22,284 - root - INFO - Collected 9349 works
2025-05-16 15:49:22,285 - root - INFO - Querying OpenAlex for '((parenting AND engagement) OR (parenting AND engage) OR (parenting AND intervention) 

In [11]:
_data_df = data_df.drop_duplicates("id")
len(_data_df)

39155

In [12]:
# check difference in size after deduplication
len(data_df) - len(_data_df)

22199

In [13]:
# manually uploaded to s3://discovery-iss/data/afs_scanning/afs_open_alex_scan_parenting_interventions.csv
_data_df.to_csv("afs_open_alex_scan_parenting_interventions.csv", index=False)

In [ ]:
((child OR children) AND (speech OR language OR communication) AND (intervention))